# 01 - Exploratory Data Analysis (EDA)

This notebook contains the exploratory data analysis for the Retail Demand Forecasting dataset. We will analyze the basic structure, missing values, duplicates, column data types, target variable distribution (`Sales`), feature relationships, correlation, and time-based sales trends.

## 1. Introduction

The goal of this project is to build a machine learning system capable of forecasting future daily sales for retail stores. Accurate demand forecasting helps optimize inventory levels, minimizing storage costs while preventing stockouts.

In this notebook, we perform the initial Exploratory Data Analysis (EDA) on the primary historical training dataset (`train.csv`) to understand its structure, quality, and statistical properties.

## 2. Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set matplotlib/seaborn style
sns.set_theme(style="whitegrid")

In [ ]:
# Load historical training data and store metadata
train = pd.read_csv("../data/raw/train.csv", low_memory=False)
store = pd.read_csv("../data/raw/store.csv")

# Filter training set to only open days with sales for accurate EDA
df_active = train[(train['Open'] == 1) & (train['Sales'] > 0)].copy()
df = pd.merge(df_active, store, on='Store', how='left')

# Format date columns
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week

df.head()

## 3. Data Quality Checks

We check the basic properties of the dataset: shape, missing values, duplicate records, data types, and potential outliers.

In [ ]:
# 1. Dimensions: How many rows and columns?
print(f"Dataset Shape (Active Days Merged): {df.shape[0]:,} rows, {df.shape[1]} columns\n")

# 2. Missing Values: Which columns have missing values?
missing = df.isnull().sum()
missing_cols = missing[missing > 0]
if not missing_cols.empty:
    print("Columns with missing values:")
    for col, count in missing_cols.items():
        print(f"  - {col}: {count:,} nulls ({count/len(df)*100:.2f}%)")
else:
    print("No missing values found in any columns.")
print()

# 3. Duplicate Rows: Are there any duplicates?
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates:,}\n")

# 4. Datatypes: Are they correct?
print("Column Datatypes:")
print(df.dtypes)

In [ ]:
# 5. Outliers (using IQR method for Sales and Customers)
q1_sales = df['Sales'].quantile(0.25)
q3_sales = df['Sales'].quantile(0.75)
iqr_sales = q3_sales - q1_sales
upper_sales = q3_sales + 1.5 * iqr_sales
lower_sales = max(0, q1_sales - 1.5 * iqr_sales)
outliers_sales_df = df[(df['Sales'] < lower_sales) | (df['Sales'] > upper_sales)]
print(f"Outliers in Sales (Outside [{lower_sales:.1f}, {upper_sales:.1f}]):")
print(f"  - Outlier count: {len(outliers_sales_df):,} ({len(outliers_sales_df)/len(df)*100:.2f}%)")
print(f"  - Max outlier: {df['Sales'].max():,}\n")

q1_cust = df['Customers'].quantile(0.25)
q3_cust = df['Customers'].quantile(0.75)
iqr_cust = q3_cust - q1_cust
upper_cust = q3_cust + 1.5 * iqr_cust
lower_cust = max(0, q1_cust - 1.5 * iqr_cust)
outliers_cust_df = df[(df['Customers'] < lower_cust) | (df['Customers'] > upper_cust)]
print(f"Outliers in Customers (Outside [{lower_cust:.1f}, {upper_cust:.1f}]):")
print(f"  - Outlier count: {len(outliers_cust_df):,} ({len(outliers_cust_df)/len(df)*100:.2f}%)")
print(f"  - Max outlier: {df['Customers'].max():,}")

## 4. Target Variable Analysis

We focus on the target variable `Sales` to explore its distribution characteristics (skewness, kurtosis) and visualize the shape of the data using a histogram and boxplot.

In [ ]:
sales = df['Sales']
print("Sales Statistics:")
print(f"  - Mean:               {sales.mean():.2f}")
print(f"  - Median:             {sales.median():.2f}")
print(f"  - Max:                {sales.max():,}")
print(f"  - Min:                {sales.min():,}")
print(f"  - Standard Deviation: {sales.std():.2f}")
print(f"  - Skewness:           {sales.skew():.4f}")
print(f"  - Kurtosis:           {sales.kurt():.4f}")

In [ ]:
# Plot Distribution of Sales
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Histogram
sns.histplot(df['Sales'], bins=50, kde=True, ax=axes[0], color="#2b5c8f")
axes[0].set_title("Sales Distribution (Histogram)", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Sales", fontsize=12)
axes[0].set_ylabel("Count", fontsize=12)

# Boxplot
sns.boxplot(x=df['Sales'], ax=axes[1], color="#4f81bd")
axes[1].set_title("Sales Distribution (Boxplot)", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Sales", fontsize=12)

plt.tight_layout()
plt.show()

## 5. Feature Relationships Analysis

Let's explore how different features (stores, calendar parameters, promotions, holidays, competition) relate to daily sales.

In [ ]:
# Create plotting layout
fig, axes = plt.subplots(4, 2, figsize=(16, 20))

# 1. Store: Top 10 stores by average sales
top10_stores = df.groupby('Store')['Sales'].mean().sort_values(ascending=False).head(10)
sns.barplot(x=top10_stores.index, y=top10_stores.values, ax=axes[0, 0], palette="Blues_r", order=top10_stores.index)
axes[0, 0].set_title("Top 10 Stores by Average Sales", fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel("Store ID")
axes[0, 0].set_ylabel("Average Sales")

# 2. DayOfWeek: Average sales by weekday
day_sales = df.groupby('DayOfWeek')['Sales'].mean()
sns.barplot(x=day_sales.index, y=day_sales.values, ax=axes[0, 1], palette="Blues")
axes[0, 1].set_title("Average Sales by Day of Week", fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel("Day of Week (1=Mon, 7=Sun)")
axes[0, 1].set_ylabel("Average Sales")

# 3. Promo: Does promotion increase sales?
promo_sales = df.groupby('Promo')['Sales'].mean()
sns.barplot(x=promo_sales.index, y=promo_sales.values, ax=axes[1, 0], palette="Blues")
axes[1, 0].set_title("Average Sales: Promo vs No Promo", fontsize=12, fontweight='bold')
axes[1, 0].set_xticklabels(["No Promo (0)", "Promo (1)"])
axes[1, 0].set_xlabel("Promotion Active")
axes[1, 0].set_ylabel("Average Sales")

# 4. StateHoliday: Holiday vs non-holiday sales (when store is open)
df['StateHoliday'] = df['StateHoliday'].astype(str).str.strip()
state_sales = df.groupby('StateHoliday')['Sales'].mean()
sns.barplot(x=state_sales.index, y=state_sales.values, ax=axes[1, 1], palette="Blues")
axes[1, 1].set_title("Average Sales by State Holiday (Open Days Only)", fontsize=12, fontweight='bold')
axes[1, 1].set_xticklabels(["None (0)", "Public (a)", "Easter (b)", "Christmas (c)"])
axes[1, 1].set_xlabel("Holiday Type")
axes[1, 1].set_ylabel("Average Sales")

# 5. SchoolHoliday: Does school holiday closure affect sales?
school_sales = df.groupby('SchoolHoliday')['Sales'].mean()
sns.barplot(x=school_sales.index, y=school_sales.values, ax=axes[2, 0], palette="Blues")
axes[2, 0].set_title("Average Sales: School Holiday vs No School Holiday", fontsize=12, fontweight='bold')
axes[2, 0].set_xticklabels(["No (0)", "Yes (1)"])
axes[2, 0].set_xlabel("School Holiday")
axes[2, 0].set_ylabel("Average Sales")

# 6. Customers: Correlation with Sales
sample_df = df.sample(10000, random_state=42)
sns.scatterplot(data=sample_df, x="Customers", y="Sales", alpha=0.3, color="#2b5c8f", ax=axes[2, 1])
axes[2, 1].set_title("Customers vs Sales (10k Sample Scatter)", fontsize=12, fontweight='bold')
axes[2, 1].set_xlabel("Number of Customers")
axes[2, 1].set_ylabel("Sales")

# 7. CompetitionDistance: Impact of competitor distance
df['CompDistanceBin'] = pd.qcut(df['CompetitionDistance'].dropna(), q=10, labels=False)
comp_bin_sales = df.groupby('CompDistanceBin')['Sales'].mean()
sns.lineplot(x=comp_bin_sales.index, y=comp_bin_sales.values, marker="o", color="#2b5c8f", ax=axes[3, 0])
axes[3, 0].set_title("Average Sales by Competition Distance Decile", fontsize=12, fontweight='bold')
axes[3, 0].set_xlabel("Distance Decile (0 = Closest, 9 = Furthest)")
axes[3, 0].set_ylabel("Average Sales")

# 8. Promo2: Does long-running promotion increase sales?
promo2_sales = df.groupby('Promo2')['Sales'].mean()
sns.barplot(x=promo2_sales.index, y=promo2_sales.values, ax=axes[3, 1], palette="Blues")
axes[3, 1].set_title("Average Sales: Promo2 vs No Promo2", fontsize=12, fontweight='bold')
axes[3, 1].set_xticklabels(["No Promo2 (0)", "Promo2 (1)"])
axes[3, 1].set_xlabel("Promo2 Participating")
axes[3, 1].set_ylabel("Average Sales")

plt.tight_layout()
plt.show()

### Answers to Feature Relationship Questions:
- **Store**: Stores like `817`, `262`, `1114`, `251`, and `842` generate the highest average sales (often exceeding 18k, compared to the overall average active sales of 6.95k).
- **DayOfWeek**: Monday performs best ($\approx$ 8,216) among standard weekdays. While Sundays show high average sales ($\approx$ 8,224) *when open*, only a small fraction of stores (3,593 out of 144,730 total Sunday observations) actually remain open.
- **Promo**: Active short-term promotions increase average sales significantly, resulting in a **38.77% lift** (from 5,929 to 8,228).
- **StateHoliday**: While most stores close on state holidays, the few stores that stay open generate exceptionally high average sales ($\approx$ 9.8k on Easter and Christmas, $\approx$ 8.5k on Public Holidays compared to 6.9k baseline), representing high demand for open stores.
- **SchoolHoliday**: Yes, it slightly increases average daily sales (from 6,897 to 7,200, a **+4.4% increase**), representing slightly higher shopping traffic when children are out of school.
- **Customers**: Highly correlated with Sales (Pearson Correlation of **0.8236**). It is the single strongest numeric predictor of daily store sales.
- **CompetitionDistance**: Nearest competitor distance does not show a strong linear relationship (Correlation **-0.0365**). However, looking at the deciles, stores with extremely close competition (decile 0) actually exhibit slightly higher average sales, possibly due to prime high-traffic locations shared by competitors.
- **Promo2**: Interestingly, stores participating in the continuous `Promo2` program show *lower* active daily sales on average (6,558 vs 7,350 without Promo2), suggesting it might be implemented in historically lower-performing stores or that continuous promos lead to customer promo-fatigue.

## 6. Correlation Analysis

Let's look at the correlation matrix and heatmap for numerical variables to find potential predictors and multicollinearity issues.

In [ ]:
plt.figure(figsize=(10, 8))
numeric_cols = ['Sales', 'Customers', 'DayOfWeek', 'Promo', 'SchoolHoliday', 'CompetitionDistance', 'Promo2', 'Year', 'Month', 'Week']
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap="Blues", fmt=".2f", linewidths=0.5, cbar=True, square=True)
plt.title("Correlation Matrix Heatmap (Numeric Features)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Answers to Correlation Questions:
- **Which features are most correlated with Sales?**
  - `Customers` is heavily correlated at **0.82**.
  - `Promo` (short-term promotion) is moderately correlated at **0.37**.
  - `DayOfWeek` has a negative correlation of **-0.18** because sales drop towards the weekend (especially Saturday).
- **Are there highly correlated features (multicollinearity)?**
  - Yes. `Month` and `Week` (week of year) show an extremely high correlation of **0.97**. During modeling, we should consider dropping one or combining them to avoid multicollinearity issues.

## 7. Time Analysis

We analyze trends and seasonality patterns over time (yearly, monthly, weekly, and total aggregate sales).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Monthly total sales trend over time
monthly_sales_time = df.groupby(['Year', 'Month'])['Sales'].sum().reset_index()
monthly_sales_time['Date'] = pd.to_datetime(monthly_sales_time[['Year', 'Month']].assign(Day=1))
sns.lineplot(data=monthly_sales_time, x="Date", y="Sales", marker="o", color="#2b5c8f", ax=axes[0, 0])
axes[0, 0].set_title("Total Sales Trend over Time (Monthly)", fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("Total Monthly Sales")

# 2. Sales by Month (Seasonality)
monthly_avg = df.groupby('Month')['Sales'].mean()
sns.barplot(x=monthly_avg.index, y=monthly_avg.values, palette="Blues", ax=axes[0, 1])
axes[0, 1].set_title("Average Sales by Month (Seasonality)", fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel("Month")
axes[0, 1].set_ylabel("Average Sales")

# 3. Sales by Year (Trend)
yearly_avg = df.groupby('Year')['Sales'].mean()
sns.barplot(x=yearly_avg.index, y=yearly_avg.values, palette="Blues", ax=axes[1, 0])
axes[1, 0].set_title("Average Sales by Year (Trend)", fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel("Year")
axes[1, 0].set_ylabel("Average Sales")

# 4. Sales by Week of Year
weekly_avg = df.groupby('Week')['Sales'].mean()
sns.lineplot(x=weekly_avg.index, y=weekly_avg.values, color="#2b5c8f", ax=axes[1, 1])
axes[1, 1].set_title("Average Sales by Week of Year", fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel("Week of Year")
axes[1, 1].set_ylabel("Average Sales")

plt.tight_layout()
plt.show()

### Answers to Time Analysis Questions:
- **Is there seasonality?**
  - **Yes, strong yearly seasonality is visible.** There is a sharp sales surge every year in **December** (averaging 8,608 per active day compared to the ~6.9k yearly average), representing the end-of-year holiday shopping season, followed by a dramatic drop in **January** (averaging 6,564) and **February** (6,589).
- **Is there an upward/downward trend?**
  - **Yes, a general upward trend.** Average sales per active day grew steadily from **6,814 in 2013** to **7,026 in 2014**, and **7,088 in 2015**. Total sales volume fluctuates, but average transactions are increasing over the years.

## 8. Business Insights

Here are the actionable, data-driven business insights derived from this analysis:

- 📈 **Promotions Drive High Volume**: Running short-term promotions (`Promo`) increases active store sales by **38.77%** on average. Inventory and staffing should be scaled up during promotion weeks to meet this surge.
- 🚪 **Sunday Opportunities & Baseline**: Standard Sundays generate the lowest total sales because **97.5% of stores remain closed**. However, the few stores that do remain open on Sundays (such as transit hubs or tourist areas) generate massive average turnovers ($pprox$ 8,224), indicating high demand during weekends that could be capitalized on if local regulations permit.
- 👥 **Customers are the Primary Engine**: Store sales are heavily correlated with customer count ($r = 0.8236$). Improving customer footfall (through marketing, store layout, or loyalty programs) directly translates to revenue increases.
- 🎄 **End-of-Year Seasonal Surge**: December generates **24.5% higher average sales** compared to other months due to seasonal holiday shopping. Supply chains must prepare weeks in advance for this predictable peak.
- 🏪 **Continuous Promotion Fatigue**: Stores implementing continuous long-running promotions (`Promo2`) actually see **10.8% lower sales** than stores that don't. Continuous promos may dilute brand value or reduce purchase urgency; short-term promo strategies should be prioritized instead.